# L07 Numba & Multiprocessing - Exam Exercises
Yagmur Yesilyurt

## Q1: The stock market (Markov Chain + numba)

In [ ]:
import numpy as np
import time
from numba import njit

# states: 0=bull, 1=bear, 2=recession
# transition matrix from the diagram
# row = current state, col = next state
transition = np.array([
    [0.9,  0.075, 0.025],  # from bull
    [0.15, 0.8,   0.05 ],  # from bear
    [0.25, 0.25,  0.5  ],  # from recession
])

In [ ]:
# pure python version
def simulate_python(n_steps, transition):
    state = np.random.randint(0, 3)
    counts = [0, 0, 0]
    for _ in range(n_steps):
        state = np.random.choice(3, p=transition[state])
        counts[state] += 1
    return counts

In [ ]:
# numba version -- same logic, but compiled
@njit
def simulate_numba(n_steps, transition):
    state = np.random.randint(0, 3)
    counts = np.zeros(3, dtype=np.int64)
    for _ in range(n_steps):
        r = np.random.random()
        # manually pick next state from cumulative probabilities
        cumsum = 0.0
        for j in range(3):
            cumsum += transition[state, j]
            if r < cumsum:
                state = j
                break
        counts[state] += 1
    return counts

In [ ]:
N = 1_000_000

# warm up numba (first call compiles)
simulate_numba(10, transition)

t0 = time.time()
counts_py = simulate_python(N, transition)
t_py = time.time() - t0

t0 = time.time()
counts_nb = simulate_numba(N, transition)
t_nb = time.time() - t0

print(f"Pure Python: {t_py:.3f} s")
print(f"Numba:       {t_nb:.3f} s")
print(f"Speedup:     {t_py/t_nb:.1f}x")
print()
print("Fraction of days in each state (numba):")
labels = ["Bull", "Bear", "Recession"]
for label, c in zip(labels, counts_nb):
    print(f"  {label:10s}: {c/N:.3f}")

The fractions should converge to the stationary distribution of the Markov chain. Numba is typically 50-200x faster than pure Python for this kind of loop-heavy code.

## Q2: Consistent plotting (decorator)

In [ ]:
import matplotlib.pyplot as plt
import functools

def myplot(func):
    """
    Decorator that sets up a publication-ready matplotlib figure before the
    plotting function runs, then saves the result to a PDF automatically.
    The decorated function just needs to return the figure object.
    """
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # apply my preferred global style before the function does anything
        plt.rcParams.update({
            "font.size": 12,
            "axes.labelsize": 13,
            "axes.titlesize": 13,
            "xtick.direction": "in",
            "ytick.direction": "in",
            "xtick.top": True,
            "ytick.right": True,
            "legend.frameon": False,
            "figure.dpi": 120,
        })

        fig = func(*args, **kwargs)

        # save to pdf with the function name as filename
        filename = f"{func.__name__}.pdf"
        fig.savefig(filename, bbox_inches="tight")
        print(f"saved to {filename}")
        return fig

    return wrapper

In [ ]:
# example: any plotting function just gets @myplot on top
# no need to set fontsizes or call savefig manually ever again

@myplot
def plot_sine_wave():
    x = np.linspace(0, 2*np.pi, 300)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(x, np.sin(x), label=r"$\sin(x)$")
    ax.plot(x, np.cos(x), label=r"$\cos(x)$", linestyle="--")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title("Trig functions")
    ax.legend()
    return fig

fig = plot_sine_wave()
plt.show()

Now every plotting function in a paper just needs `@myplot` and it will automatically have consistent style and be saved to PDF — no copy-pasting `rcParams` or `savefig` calls everywhere.